# Grass / Ogerpon — BC training notebook

Dataset: `dataset_grass_ogerpon.npz` — the exact 60-card Ogerpon/Mega Kangaskhan/
Latias shell used by the #2 ladder team, built from public replays.

What this notebook does:
1. Trains a small **value net** (state -> game win probability).
2. Trains a **policy prior** (legal options -> ranking) on winning-game,
   single-selection decisions. ATTACH/target decisions are zero-weighted.
3. Exports pure-NumPy `.npz` weights and runs a PyTorch -> NumPy parity check.

No submission is produced here. Add the dataset as a private Kaggle input and run all.


In [ ]:
import json, math, os, time, glob
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, 'device', device)

OUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('.')
candidates = list(Path('/kaggle/input').rglob('dataset_grass_ogerpon.npz'))
if not candidates:
    candidates = list(Path('.').rglob('dataset_grass_ogerpon.npz'))
if not candidates:
    raise FileNotFoundError('dataset_grass_ogerpon.npz not found; attach it as an input')
DATA_PATH = candidates[0]
print('dataset:', DATA_PATH)


In [ ]:
d = np.load(DATA_PATH)
state = d['state'].astype(np.float32)
options = d['options']
card_ids = d['card_ids'].astype(np.int16)
mask = d['mask'].astype(np.float32)
label = d['label'].astype(np.int64)
game_win = d['game_win'].astype(np.float32)
split = d['split'].astype(np.int8)
selected_type = d['selected_type'].astype(np.int16)
context = d['context'].astype(np.int16)
max_count = d['max_count'].astype(np.int16)
action_len = d['action_length'].astype(np.int16)
train_idx = np.nonzero(split == 0)[0]
val_idx = np.nonzero(split == 1)[0]
print('rows', len(state), 'train', len(train_idx), 'val', len(val_idx))
print('state', state.shape, 'options', options.shape, 'win mean', float(game_win.mean()))
print('single-select labels', int((label >= 0).sum()))


## 1. Value net

Small MLP on the 78-dim public-state vector. Early stopping on validation BCE.


In [ ]:
class ValueNet(nn.Module):
    def __init__(self, state_dim, hidden=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def vmetrics(y, logit):
    p = 1 / (1 + np.exp(-logit))
    bce = float(np.mean(np.logaddexp(0, logit) - y * logit))
    acc = float(((p >= .5) == (y >= .5)).mean())
    brier = float(np.mean((p - y) ** 2))
    order = np.argsort(p); ranks = np.empty_like(order, dtype=np.float64); ranks[order] = np.arange(1, len(p) + 1)
    pos = float(y.sum()); neg = float(len(y) - pos)
    auc = float((ranks[y >= .5].sum() - pos * (pos + 1) / 2) / (pos * neg)) if pos > 0 and neg > 0 else .5
    return bce, acc, auc, brier

v_mean = state[train_idx].mean(0)
v_std = state[train_idx].std(0); v_std[v_std < 1e-4] = 1.0
vnet = ValueNet(state.shape[1]).to(device)
vopt = optim.AdamW(vnet.parameters(), lr=3e-4, weight_decay=1e-3)
vloss = nn.BCEWithLogitsLoss()
best = (1e9, None, 0); bad = 0

@torch.no_grad()
def vinfer(index, batch=4096):
    vnet.eval(); out = []
    for start in range(0, len(index), batch):
        ii = index[start:start+batch]
        x = torch.from_numpy(((state[ii] - v_mean) / v_std).astype(np.float32)).to(device)
        out.append(vnet(x).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)

for epoch in range(1, 41):
    vnet.train(); order = train_idx.copy(); np.random.shuffle(order); total = 0.0; count = 0
    for start in range(0, len(order), 1024):
        ii = order[start:start+1024]
        x = torch.from_numpy(((state[ii] - v_mean) / v_std).astype(np.float32)).to(device)
        yy = torch.from_numpy(game_win[ii]).to(device)
        vopt.zero_grad(set_to_none=True)
        loss = vloss(vnet(x), yy)
        loss.backward(); vopt.step()
        total += float(loss.item()) * len(ii); count += len(ii)
    m = vmetrics(game_win[val_idx], vinfer(val_idx))
    print(f'epoch {epoch:2d} train_bce={total/max(1,count):.4f} val_bce={m[0]:.4f} acc={m[1]:.4f} auc={m[2]:.4f}')
    if m[0] < best[0] - 1e-12:
        best = (m[0], {k: v.detach().cpu().clone() for k, v in vnet.state_dict().items()}, epoch); bad = 0
    else:
        bad += 1
        if bad >= 8:
            break
vnet.load_state_dict(best[1])
print('best epoch', best[2], vmetrics(game_win[val_idx], vinfer(val_idx)))

# Export + PyTorch -> NumPy parity
sample = val_idx[:1000]
torch_logit = vinfer(sample)
x = (state[sample] - v_mean) / v_std
w0 = vnet.net[0].weight.detach().cpu().numpy().astype(np.float32); b0 = vnet.net[0].bias.detach().cpu().numpy().astype(np.float32)
w1 = vnet.net[3].weight.detach().cpu().numpy().astype(np.float32); b1 = vnet.net[3].bias.detach().cpu().numpy().astype(np.float32)
w2 = vnet.net[6].weight.detach().cpu().numpy().astype(np.float32); b2 = vnet.net[6].bias.detach().cpu().numpy().astype(np.float32)
h = np.maximum(0, x @ w0.T + b0); h = np.maximum(0, h @ w1.T + b1); np_logit = (h @ w2.T + b2).reshape(-1)
print('value parity max diff', float(np.max(np.abs(np_logit - torch_logit))))
np.savez_compressed(OUT/'value_model_grass.npz', state_mean=v_mean, state_std=v_std, w0=w0, b0=b0, w1=w1, b1=b1, w2=w2, b2=b2)
print('wrote', OUT/'value_model_grass.npz')


## 2. Policy prior

State-conditioned pointer scorer with one attention block and card-id embeddings.
Training rows: winner games, single-selection decisions. ATTACH/target contexts
are kept out of the loss (zero sample weight); the rule/search layer owns those.


In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, state_dim, option_dim, embed=32, attn=64, hidden=128, dropout=.15):
        super().__init__()
        self.card_embedding = nn.Embedding(2048, embed, padding_idx=0)
        self.state_proj = nn.Linear(state_dim, attn)
        self.option_proj = nn.Linear(option_dim + 2 * embed, attn)
        self.q = nn.Linear(attn, attn); self.k = nn.Linear(attn, attn); self.v = nn.Linear(attn, attn)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(2 * attn, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.scale = math.sqrt(attn)
    def forward(self, s, o, cid, m):
        actor = self.card_embedding(cid[..., 0].long()); target = self.card_embedding(cid[..., 1].long())
        tok = self.dropout(self.option_proj(torch.cat([o, actor, target], -1)))
        st = self.state_proj(s)
        score = torch.matmul(self.q(st).unsqueeze(1), self.k(tok).transpose(-1, -2)).squeeze(1) / self.scale
        score = score.masked_fill(m <= 0, -1e9)
        att = torch.softmax(score, -1)
        ctx = torch.matmul(att.unsqueeze(1), self.v(tok)).squeeze(1)
        joint = torch.cat([tok, ctx.unsqueeze(1).expand(-1, tok.size(1), -1)], -1)
        return self.head(joint).squeeze(-1).masked_fill(m <= 0, -1e9)

def masked_ce(logits, y, m, w=None):
    ok = (y >= 0) & (m.sum(-1) > 0)
    if not bool(ok.any()): return logits.sum() * 0
    logits, y = logits[ok], y[ok]
    if w is None: return nn.functional.cross_entropy(logits, y)
    losses = nn.functional.cross_entropy(logits, y, reduction='none'); ww = w[ok]
    return (losses * ww).sum() / ww.sum().clamp_min(1e-8)

label_in = (label >= 0) & (label < options.shape[1])
label_ok = np.zeros(len(label), dtype=bool)
rows = np.nonzero(label_in)[0]
label_ok[rows] = mask[rows, label[rows]] > 0
usable = label_in & label_ok & (max_count == 1) & (action_len == 1) & (game_win >= .5) & (mask.sum(1) > 1)
zero_weight = (selected_type == 8) | np.isin(context, [21,22,26,27,28,29,30])
ptrain = np.nonzero(usable & (split == 0))[0]
pval = np.nonzero(usable & (split == 1))[0]
print('policy train', len(ptrain), 'val', len(pval))

p_mean = state[ptrain].mean(0); p_std = state[ptrain].std(0); p_std[p_std < 1e-4] = 1.0
valid_opt = np.asarray(mask[ptrain] > 0)
chosen_opt = options[ptrain][valid_opt].astype(np.float32)
o_mean = chosen_opt.mean(0); o_std = chosen_opt.std(0); o_std[o_std < 1e-4] = 1.0
pweight = np.where(zero_weight, 0.0, 1.0).astype(np.float32)

pnet = PolicyNet(state.shape[1], options.shape[2]).to(device)
popt = optim.AdamW(pnet.parameters(), lr=1e-3, weight_decay=1e-4)

@torch.no_grad()
def peval(index, batch=512):
    pnet.eval(); total = 0.0; correct = 0; top3 = 0; count = 0
    for start in range(0, len(index), batch):
        ii = index[start:start+batch]
        s = torch.from_numpy(((state[ii] - p_mean) / p_std).astype(np.float32)).to(device)
        o = torch.from_numpy(((options[ii].astype(np.float32) - o_mean) / o_std)).to(device)
        c = torch.from_numpy(card_ids[ii].astype(np.int64)).to(device)
        m = torch.from_numpy(mask[ii]).to(device); y = torch.from_numpy(label[ii]).to(device)
        logits = pnet(s, o, c, m)
        total += float(masked_ce(logits, y, m).item()) * len(ii); count += len(ii)
        pred = logits.argmax(-1).cpu().numpy(); correct += int((pred == label[ii]).sum())
        top = np.argsort(-logits.cpu().numpy(), axis=1)[:, :3]; top3 += int(np.any(top == label[ii, None], axis=1).sum())
    return total / max(1, count), correct / max(1, count), top3 / max(1, count)

best = (-1.0, 1e9, None, 0); bad = 0
for epoch in range(1, 31):
    pnet.train(); order = ptrain.copy(); np.random.shuffle(order); total = 0.0; count = 0
    for start in range(0, len(order), 256):
        ii = order[start:start+256]
        s = torch.from_numpy(((state[ii] - p_mean) / p_std).astype(np.float32)).to(device)
        o = torch.from_numpy(((options[ii].astype(np.float32) - o_mean) / o_std)).to(device)
        c = torch.from_numpy(card_ids[ii].astype(np.int64)).to(device)
        m = torch.from_numpy(mask[ii]).to(device); y = torch.from_numpy(label[ii]).to(device)
        w = torch.from_numpy(pweight[ii]).to(device)
        popt.zero_grad(set_to_none=True)
        loss = masked_ce(pnet(s, o, c, m), y, m, w)
        loss.backward(); popt.step()
        total += float(loss.item()) * len(ii); count += len(ii)
    ce, top1, top3 = peval(pval)
    print(f'epoch {epoch:2d} train_ce={total/max(1,count):.4f} val_ce={ce:.4f} top1={top1:.4f} top3={top3:.4f}')
    improved = top1 > best[0] + 1e-12 or (abs(top1 - best[0]) <= 1e-12 and ce < best[1] - 1e-12)
    if improved:
        best = (top1, ce, {k: v.detach().cpu().clone() for k, v in pnet.state_dict().items()}, epoch); bad = 0
    else:
        bad += 1
        if bad >= 6: break
pnet.load_state_dict(best[2])
print('best epoch', best[3], peval(pval))

# NumPy parity on 200 rows
sample = pval[:200]
with torch.no_grad():
    st = torch.from_numpy(((state[sample] - p_mean) / p_std).astype(np.float32)).to(device)
    ot = torch.from_numpy(((options[sample].astype(np.float32) - o_mean) / o_std)).to(device)
    ct = torch.from_numpy(card_ids[sample].astype(np.int64)).to(device)
    mt = torch.from_numpy(mask[sample]).to(device)
    logits_t = pnet(st, ot, ct, mt).cpu().numpy()
w = {k: v.detach().cpu().numpy().astype(np.float32) for k, v in pnet.state_dict().items()}
s = ((state[sample] - p_mean) / p_std).astype(np.float32)
o = ((options[sample].astype(np.float32) - o_mean) / o_std)
cid = card_ids[sample]
emb = w['card_embedding.weight']
inp = np.concatenate([o, emb[cid[:, :, 0]], emb[cid[:, :, 1]]], -1)
tok = inp @ w['option_proj.weight'].T + w['option_proj.bias']
stt = s @ w['state_proj.weight'].T + w['state_proj.bias']
q = stt @ w['q.weight'].T + w['q.bias']
k = tok @ w['k.weight'].T + w['k.bias']
v = tok @ w['v.weight'].T + w['v.bias']
score = np.einsum('bd,bnd->bn', q, k) / math.sqrt(tok.shape[-1])
score = np.where(mask[sample] > 0, score, -1e9); score -= score.max(1, keepdims=True)
att = np.exp(score); att = np.where(mask[sample] > 0, att, 0); att /= np.maximum(att.sum(1, keepdims=True), 1e-12)
ctx = np.einsum('bn,bnd->bd', att, v)
joint = np.concatenate([tok, np.broadcast_to(ctx[:, None, :], tok.shape)], -1)
hidden = np.maximum(0, joint @ w['head.0.weight'].T + w['head.0.bias'])
logits_np = (hidden @ w['head.3.weight'].T + w['head.3.bias']).reshape(logits_t.shape)
logits_np = np.where(mask[sample] > 0, logits_np, -1e9)
print('policy parity max diff', float(np.max(np.abs(logits_np - logits_t))))
np.savez_compressed(OUT/'policy_model_grass.npz',
    state_mean=p_mean, state_std=p_std, option_mean=o_mean, option_std=o_std, **w)
print('wrote', OUT/'policy_model_grass.npz')


## 3. Next step

The exported `.npz` files are pure NumPy inference artifacts. To use them in an
agent, the runtime must call `state_vector(..., profile='grass')` and the same
option/card-id featurizer, then load these weights. Keep the rule core as the
legal fallback and use the policy only as a candidate prior.

No Kaggle submission is produced by this notebook.
